In [ ]:
import os, glob, subprocess
if not os.path.exists('/kaggle/working/Basma'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/us3ph/Basma.git', '/kaggle/working/Basma'], check=True)
os.chdir('/kaggle/working/Basma'); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
# every data build (old + v4 + the five v5 OpenFake parts), and the NTIRE public test images (downloaded to /tmp: never part of any output)
import subprocess
from huggingface_hub import hf_hub_download
mans = sorted(m for m in glob.glob('/kaggle/input/**/manifest_*.jsonl', recursive=True) if '/basma-data-' in m)
dirs = sorted({os.path.dirname(m) for m in mans}); print('DATA DIRS', dirs)
assert len(dirs) == 13, f'expected 13 data builds, found {mans}'
R = 'deepfakesMSU/NTIRE-RobustAIGenDetection-test-public'
z = hf_hub_download(R, 'test_images.zip', repo_type='dataset', local_dir='/tmp/ntire')
subprocess.run(['unzip', '-q', '-o', z, '-d', '/tmp/ntire/img'], check=True)
args = ['python', 'scripts/leak_guard.py', '--data-dirs', *dirs, '--ntire', '/tmp/ntire/img', '--out', '/kaggle/working', '--workers', '4']
r = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True); print(r.stdout[-8000:])
assert r.returncode == 0, 'leak guard failed'


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('/kaggle/working/LEAKGUARD.md').read()))


In [ ]:
# the code dir must not be part of the output dataset
import shutil, os, subprocess
os.chdir('/kaggle/working'); shutil.rmtree('/kaggle/working/Basma', ignore_errors=True)
print(subprocess.run(['du', '-sh', '/kaggle/working'], capture_output=True, text=True).stdout); print(sorted(os.listdir('/kaggle/working')))
